# StatsGeeks: Coordinate-RF final solution

Four-class building-age transfer from Madrid to Amsterdam. Metric: macro-F1. This notebook uses the packaged trusted, preprocessed target-pool artifact. It is **not self-contained raw-data training**. Source organizer parquet files and source-fitted preprocessing are external for a raw rebuild. No competitor material is used.

Coordinate-RF predicts four building-age classes for Amsterdam using labelled Madrid source data and a small class-balanced Amsterdam support set. The target is macro-F1, which weights each class equally. We evaluate 5, 25, 50, 100 and 200 support pixels per class.

Our source branch augments 60 spectral inputs with local 3-by-3 lattice averages, aligns source and unlabelled target covariance, and refines class-conditional alignment using source-model pseudo-labels, never query truth. Two source Random Forests produce a target probability prior. The local branch adaptively whitens 60 inputs using full unlabelled target covariance with budget-dependent diagonal shrinkage, appends two standardized lattice coordinates, and fits a 200-tree, balanced Random Forest to support labels only. Its probabilities are blended with the source prior using weight 20/(20+4b), then smoothed over nine spatial neighbours with Gaussian weights.

The locked random-pixel results, each from 200 paired episodes, are 0.650253, 0.687664, 0.708436, 0.729839 and 0.749299 macro-F1 across the five budgets. At 200 shots the population SD is 0.005716. Matched gains are 0.016262 over ASTRA, 0.091697 over EXP-F and 0.126459 over EXP-010. All 200 paired episodes beat ASTRA at that budget. The five-shot advantage is negligible.

This is transductive, full-pool, coordinate-aware inference, not an inductive unseen-city claim. The packaged artifact is bound to the supplied ordered Amsterdam pool. Rebuilding for another pool requires organizer feature data, feature order, source-fitted scaling and matching lattice coordinates. The final query population excludes the designated 800-label development bank and current support. Development labels are additional research supervision beyond each episode budget; repeated episodes are correlated. The inherited ASTRA recipe has historical audit exposure. No independent organizer-held-out validation is claimed, and protocol eligibility must be confirmed against organizer rules.

Geographic extrapolation remains a limitation: the four-direction spatial mean is 0.694216 and the worst direction is 0.644816, versus ASTRA's 0.696672 and 0.658183. Coordinate-RF is therefore selected for the stated random-pixel/full-pool use case, not universal superiority. Frozen configurations, hashes, label-free prediction replay, support/query separation and saved confusion matrices support reproducibility. No competitor code, data or predictions are included. No overnight exploratory candidate replaces the incumbent without fresh-population validation.

In [1]:
from pathlib import Path
import sys, json, pickle, hashlib
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebook': ROOT = ROOT.parent
assert (ROOT/'model/pool_state.pkl').exists(), 'Run from package root or notebook directory'
sys.path.insert(0, str(ROOT/'model'))
from agf_model import probabilities, fit_pool, CLASSES, BASE
manifest = json.loads((ROOT/'model/INFERENCE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((ROOT/'model'/name).read_bytes()).hexdigest() == expected
with (ROOT/'model/pool_state.pkl').open('rb') as f: state = pickle.load(f)
assert state['feature_names'] == manifest['feature_names']
lock = json.loads((ROOT/'model/SELECTION_LOCK.json').read_text())
assert lock['candidate'] == 'Coordinate_RF'
print('Coordinate-RF', state['raw'].shape, 'classes:', CLASSES)
print('Ordered features:', state['feature_names'])

Coordinate-RF (25992, 60) classes: [1 2 3 4]
Ordered features: ['Blue_mean', 'Green_mean', 'Red_mean', 'NIR_mean', 'SWIR1_mean', 'SWIR2_mean', 'Blue_std', 'Green_std', 'Red_std', 'NIR_std', 'SWIR1_std', 'SWIR2_std', 'NDVI_mean', 'NDBI_mean', 'UI_mean', 'MNDWI_mean', 'BSI_mean', 'NDVI_std', 'NDBI_std', 'UI_std', 'MNDWI_std', 'BSI_std', 'Blue_early_mean', 'Green_early_mean', 'Red_early_mean', 'NIR_early_mean', 'SWIR1_early_mean', 'SWIR2_early_mean', 'Blue_early_std', 'Green_early_std', 'Red_early_std', 'NIR_early_std', 'SWIR1_early_std', 'SWIR2_early_std', 'Blue_late_mean', 'Green_late_mean', 'Red_late_mean', 'NIR_late_mean', 'SWIR1_late_mean', 'SWIR2_late_mean', 'Blue_late_std', 'Green_late_std', 'Red_late_std', 'NIR_late_std', 'SWIR1_late_std', 'SWIR2_late_std', 'd_Blue_mean', 'd_Green_mean', 'd_Red_mean', 'd_NIR_mean', 'd_SWIR1_mean', 'd_SWIR2_mean', 'd_Blue_std', 'd_Green_std', 'd_Red_std', 'd_NIR_std', 'd_SWIR1_std', 'd_SWIR2_std', 'has_early_data', 'has_late_data']


## Preprocessing, feature construction and source transfer
The cached state contains raw and source-standardized 60-column features, unlabelled-pool covariance, lattice coordinates and source probabilities. Source transfer uses 120 contextual columns, covariance alignment and one pseudo-class refinement. `fit_pool` takes source labels but **no target labels**.

To rebuild from organizer preprocessed arrays, supply the exact ordered 60-column source-standardized matrices, source labels, integer coordinates, Madrid scaler mean/scale and names. The function below is intentionally not invoked without those external inputs. The archive does not invent source data or claim a raw preprocessing reproduction.

In [2]:
def rebuild_pool(source_X, source_y, source_xy, target_X, target_xy, madrid_mean, madrid_scale, feature_names):
    assert source_X.shape[1] == target_X.shape[1] == len(feature_names) == 60
    return fit_pool(source_X, source_y, source_xy, target_X, target_xy,
                    madrid_mean, madrid_scale, feature_names)
print('Frozen model parameters:', {**BASE, **lock['config']})
# Show the actual local feature construction at 200 shots/class.
from agf_model import power
budget = 200
rho = min(1., 60/(4*budget))
cov = (1-rho)*state['cov'] + rho*np.diag(np.diag(state['cov'])) + 1e-7*np.eye(60)
whitened = (state['raw']-state['raw'].mean(0)) @ power(cov, -.5)
xy = state['coords']
local_features = np.column_stack((whitened, (xy-xy.mean(0))/np.maximum(xy.std(0),1)))
assert local_features.shape[1] == 62
print('Local feature matrix:', local_features.shape, 'rho:', rho)
# Explicit model construction: the same factory is used inside probabilities().
from agf_model import forest
local_model = forest(kind='rf', leaf=1, features='sqrt', trees=200)
print('Support classifier:', local_model)
print('Fixed source-prior blend weight:', 20/(20+4*budget))

Frozen model parameters: {'kind': 'rf', 'leaf': 1, 'features': 'sqrt', 'trees': 200, 'pool': True, 'adaptive': True, 'spatial': True, 'prior': 20.0, 'rho_factor': 1.0, 'sigma': 1.0, 'neighbours': 9, 'steps': 1, 'coord': True, 'bilateral': False, 'covblend': 0.0}
Local feature matrix: (25992, 62) rho: 0.075
Support classifier: RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=4,
                       random_state=42)
Fixed source-prior blend weight: 0.024390243902439025


## Few-shot support adaptation and query prediction
Replay an existing 200-shot episode. Only support labels are loaded. Expected query **predictions**, not query truth, are used after inference for deterministic replay. The model fits RF200 on 62 columns; its fixed source blend and Gaussian graph use no query labels. User-supplied support must have exactly the chosen number of labels per class.

In [3]:
with np.load(ROOT/'model/verification_inputs/b200.npz', allow_pickle=False) as episode:
    support = episode['support'].copy()
    support_y = episode['support_labels'].copy()
    query = episode['query'].copy()
    expected = episode['expected'].copy()
assert not np.intersect1d(support, query).size
assert len(np.unique(support)) == 800
assert all(np.sum(support_y == c) == budget for c in CLASSES)
prob = probabilities(state, support, support_y, budget, lock['config'])
prediction = CLASSES[prob.argmax(1)][query]
assert np.array_equal(prediction, expected)
print('Exact reference replay:', len(prediction), 'query predictions; no query truth loaded')
print('First ten predictions:', prediction[:10])

Exact reference replay: 24392 query predictions; no query truth loaded
First ten predictions: [4 4 4 4 4 4 4 4 4 4]


## Evaluation only — never fitting or selection
The full audit truth is intentionally not packaged. Below, macro-F1 is recomputed from saved locked confusion matrices, followed by the five-budget results. Macro-F1 averages per-class F1, not accuracy. The optional scorer may be called only once predictions/configurations are fixed; it must not drive selection. SD is across 200 support-sampling episodes, not independent cities.

In [4]:
from sklearn.metrics import f1_score
def evaluate_fixed_predictions(query_truth, fixed_predictions):
    return f1_score(query_truth, fixed_predictions, labels=CLASSES, average='macro', zero_division=0)
audit = json.loads((ROOT/'model/evidence/final_complete.json').read_text())
for record in audit['records']:
    cm = np.asarray(record['confusion'])
    denom = cm.sum(0) + cm.sum(1)
    per_class = np.divide(2*np.diag(cm), denom, out=np.zeros(4), where=denom!=0)
    assert np.isclose(per_class.mean(), record['f1'], atol=1e-12)
rows = []
for b in [5,25,50,100,200]:
    result = audit['summary'][str(b)]['Coordinate_RF']
    rows.append({'shots_per_class':b, 'macro_f1':result['mean'], 'population_sd':result['population_sd'], 'episodes':result['episodes']})
print(pd.DataFrame(rows).to_string(index=False))
print(pd.read_csv(ROOT/'model/tables/spatial.csv').to_string(index=False))

 shots_per_class  macro_f1  population_sd  episodes
               5  0.650253       0.011246       200
              25  0.687664       0.012145       200
              50  0.708436       0.011277       200
             100  0.729839       0.008802       200
             200  0.749299       0.005716       200
       method           diagnostic     mean  population_sd  episodes                                                     protocol
       EXP010        x_low_to_high 0.635312       0.003101      10.0                    spatial buffered; paired within direction
       EXP010        x_high_to_low 0.596549       0.006075      10.0                    spatial buffered; paired within direction
       EXP010        y_low_to_high 0.557275       0.003819      10.0                    spatial buffered; paired within direction
       EXP010        y_high_to_low 0.681828       0.004586      10.0                    spatial buffered; paired within direction
       EXP010               x_half 0.6

## Spatial limitations and provenance
Coordinate-RF four-direction mean **0.694216**, worst **0.644816**; matched ASTRA mean **0.696672**, worst **0.658183**. All directions are reported. Random-pixel improvement is not geographic superiority.

The inherited ASTRA recipe had historical audit exposure. The designated 800-label research development bank is excluded from final queries; it is additional supervision beyond individual episode budgets. Repeated support resampling does not constitute fresh-population validation. No organizer-held-out result is claimed. The historical replay covers 1000 episodes. This overnight run also freshly refits the source branch and reconstructs all eight state arrays and five-budget predictions exactly: model/evidence/SOURCE_REFIT.json. Current package replay is reported separately in model/evidence/PREFLIGHT.json. Full-unlabelled-target/coordinate eligibility requires organizer confirmation.

See model/reports/METHODS.md, model/provenance.json, model/SELECTION_LOCK.json and model/evidence/SHA256SUMS.txt. Final selected method: **Coordinate-RF**; all overnight candidates remain exploratory unless fresh validation can be established.